In [2]:
import pandas as pd
import numpy as np

In [3]:
print("Loading NetFlix.csv ...")
df = pd.read_csv(r"C:\Users\SJ15\Downloads\NetFlix.csv")
print(f"  Loaded {len(df):,} rows x {len(df.columns)} columns")

print("\nNull counts before cleaning:")
print(df.isnull().sum())

Loading NetFlix.csv ...
  Loaded 7,787 rows x 12 columns

Null counts before cleaning:
show_id            0
type               0
title              0
director        2389
cast             718
country          507
date_added        10
release_year       0
rating             7
duration           0
genres             0
description        0
dtype: int64


In [4]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,description
0,s1,TV Show,3%,NaN,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,14-Aug-20,2020,TV-MA,4,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...
1,s10,Movie,1920,Vikram Bhatt,"Rajneesh Duggal, Adah Sharma, Indraneil Sengup...",India,15-Dec-17,2008,TV-MA,143,"Horror Movies, International Movies, Thrillers",An architect and his wife move into a castle t...
2,s100,Movie,3 Heroines,Iman Brotoseno,"Reza Rahadian, Bunga Citra Lestari, Tara Basro...",Indonesia,5-Jan-19,2016,TV-PG,124,"Dramas, International Movies, Sports Movies",Three Indonesian women break records by becomi...
3,s1000,Movie,Blue Mountain State: The Rise of Thadland,Lev L. Spiro,"Alan Ritchson, Darin Brooks, James Cade, Rob R...",United States,1-Mar-16,2016,R,90,Comedies,New NFL star Thad buys his old teammates' belo...
4,s1001,TV Show,Blue Planet II,NaN,David Attenborough,United Kingdom,3-Dec-18,2017,TV-G,1,"British TV Shows, Docuseries, Science & Nature TV",This sequel to the award-winning nature series...


In [5]:
## DATE PARSING 
# date_added is stored as "14-Aug-20" (string) -> parse to datetime
df["date_added"] = pd.to_datetime(df["date_added"], format="%d-%b-%y", errors="coerce")
df["year_added"]  = df["date_added"].dt.year.astype("Int64")
df["month_added"] = df["date_added"].dt.month.astype("Int64")
df["month_name"]  = df["date_added"].dt.strftime("%B")
print(f"\n  date_added parsed. Failed rows: {df['year_added'].isna().sum()}")


  date_added parsed. Failed rows: 98


In [6]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,description,year_added,month_added,month_name
0,s1,TV Show,3%,NaN,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,2020-08-14,2020,TV-MA,4,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...,2020,8,August
1,s10,Movie,1920,Vikram Bhatt,"Rajneesh Duggal, Adah Sharma, Indraneil Sengup...",India,2017-12-15,2008,TV-MA,143,"Horror Movies, International Movies, Thrillers",An architect and his wife move into a castle t...,2017,12,December
2,s100,Movie,3 Heroines,Iman Brotoseno,"Reza Rahadian, Bunga Citra Lestari, Tara Basro...",Indonesia,2019-01-05,2016,TV-PG,124,"Dramas, International Movies, Sports Movies",Three Indonesian women break records by becomi...,2019,1,January
3,s1000,Movie,Blue Mountain State: The Rise of Thadland,Lev L. Spiro,"Alan Ritchson, Darin Brooks, James Cade, Rob R...",United States,2016-03-01,2016,R,90,Comedies,New NFL star Thad buys his old teammates' belo...,2016,3,March
4,s1001,TV Show,Blue Planet II,NaN,David Attenborough,United Kingdom,2018-12-03,2017,TV-G,1,"British TV Shows, Docuseries, Science & Nature TV",This sequel to the award-winning nature series...,2018,12,December


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7787 entries, 0 to 7786
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   show_id       7787 non-null   object        
 1   type          7787 non-null   object        
 2   title         7787 non-null   object        
 3   director      5398 non-null   object        
 4   cast          7069 non-null   object        
 5   country       7280 non-null   object        
 6   date_added    7689 non-null   datetime64[ns]
 7   release_year  7787 non-null   int64         
 8   rating        7780 non-null   object        
 9   duration      7787 non-null   int64         
 10  genres        7787 non-null   object        
 11  description   7787 non-null   object        
 12  year_added    7689 non-null   Int64         
 13  month_added   7689 non-null   Int64         
 14  month_name    7689 non-null   object        
dtypes: Int64(2), datetime64[ns](1), int64(

In [8]:
## HANDLE NULLS
df["director"] = df["director"].fillna("Unknown")   # 2389 nulls
df["cast"]     = df["cast"].fillna("Unknown")        # 718 nulls
df["country"]  = df["country"].fillna("Unknown")     # 507 nulls
df = df.dropna(subset=["rating"])       # drop 7 rows with null rating
df = df.dropna(subset=["date_added"])   # drop rows where date parse failed
print(f"  Rows after null handling: {len(df):,}")

  Rows after null handling: 7,682


In [9]:
## DURATION SPLIT
# duration is numeric: movies = minutes (e.g. 143), TV shows = seasons (e.g. 4)
df["duration_minutes"] = np.where(df["type"] == "Movie",  df["duration"], np.nan)
df["duration_seasons"] = np.where(df["type"] == "TV Show", df["duration"], np.nan)

In [10]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,description,year_added,month_added,month_name,duration_minutes,duration_seasons
0,s1,TV Show,3%,Unknown,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,2020-08-14,2020,TV-MA,4,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...,2020,8,August,NaN,4.0
1,s10,Movie,1920,Vikram Bhatt,"Rajneesh Duggal, Adah Sharma, Indraneil Sengup...",India,2017-12-15,2008,TV-MA,143,"Horror Movies, International Movies, Thrillers",An architect and his wife move into a castle t...,2017,12,December,143.0,NaN
2,s100,Movie,3 Heroines,Iman Brotoseno,"Reza Rahadian, Bunga Citra Lestari, Tara Basro...",Indonesia,2019-01-05,2016,TV-PG,124,"Dramas, International Movies, Sports Movies",Three Indonesian women break records by becomi...,2019,1,January,124.0,NaN
3,s1000,Movie,Blue Mountain State: The Rise of Thadland,Lev L. Spiro,"Alan Ritchson, Darin Brooks, James Cade, Rob R...",United States,2016-03-01,2016,R,90,Comedies,New NFL star Thad buys his old teammates' belo...,2016,3,March,90.0,NaN
4,s1001,TV Show,Blue Planet II,Unknown,David Attenborough,United Kingdom,2018-12-03,2017,TV-G,1,"British TV Shows, Docuseries, Science & Nature TV",This sequel to the award-winning nature series...,2018,12,December,NaN,1.0


In [11]:
df['rating'].unique()

array(['TV-MA', 'TV-PG', 'R', 'TV-G', 'PG-13', 'TV-14', 'TV-Y', 'PG',
       'TV-Y7', 'NR', 'G', 'TV-Y7-FV', 'UR', 'NC-17'], dtype=object)

In [12]:
# Rating	    Meaning	                    Audience
# TV-Y	    Suitable for all children	    Kids 
# TV-Y7	    For kids 7+	                    Kids
# TV-Y7-FV	Kids 7+, fantasy violence	    Kids
# TV-G	    General audience	            Kids/Family
# G	        General audience (movies)	    Kids
# PG	    Parental guidance suggested	    Family 
# TV-PG	    Parental guidance TV	        Family
# PG-13	    Parents strongly cautioned	    Teens 
# TV-14	    Suitable for 14+	            Teens
# R	     Restricted (17+ with adult)	    Adults 
# TV-MA	    Mature audience only	        Adults
# NC-17	    Adults only (strict)	        Adults
# NR	    Not Rated	                    Unknown
# UR	    Unrated	                        Unknown

In [13]:
## AUDIENCE GROUP
rating_map = {
    "TV-G"    : "Kids",    "TV-Y"  : "Kids",
    "TV-Y7"   : "Kids",    "TV-Y7-FV": "Kids",  "G": "Kids",
    "PG"      : "Family",  "TV-PG" : "Family",
    "PG-13"   : "Teens",   "TV-14" : "Teens",
    "R"       : "Adults",  "TV-MA" : "Adults",  "NC-17": "Adults",
    "NR"      : "Unrated", "UR"    : "Unrated",
}
df["audience_group"] = df["rating"].map(rating_map).fillna("Unrated")

In [14]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,description,year_added,month_added,month_name,duration_minutes,duration_seasons,audience_group
0,s1,TV Show,3%,Unknown,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,2020-08-14,2020,TV-MA,4,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...,2020,8,August,NaN,4.0,Adults
1,s10,Movie,1920,Vikram Bhatt,"Rajneesh Duggal, Adah Sharma, Indraneil Sengup...",India,2017-12-15,2008,TV-MA,143,"Horror Movies, International Movies, Thrillers",An architect and his wife move into a castle t...,2017,12,December,143.0,NaN,Adults
2,s100,Movie,3 Heroines,Iman Brotoseno,"Reza Rahadian, Bunga Citra Lestari, Tara Basro...",Indonesia,2019-01-05,2016,TV-PG,124,"Dramas, International Movies, Sports Movies",Three Indonesian women break records by becomi...,2019,1,January,124.0,NaN,Family
3,s1000,Movie,Blue Mountain State: The Rise of Thadland,Lev L. Spiro,"Alan Ritchson, Darin Brooks, James Cade, Rob R...",United States,2016-03-01,2016,R,90,Comedies,New NFL star Thad buys his old teammates' belo...,2016,3,March,90.0,NaN,Adults
4,s1001,TV Show,Blue Planet II,Unknown,David Attenborough,United Kingdom,2018-12-03,2017,TV-G,1,"British TV Shows, Docuseries, Science & Nature TV",This sequel to the award-winning nature series...,2018,12,December,NaN,1.0,Kids


In [15]:
## CONTENT AGE
# Years between original release and when added to Netflix
df["content_age_at_add"] = df["year_added"] - df["release_year"]

In [16]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,description,year_added,month_added,month_name,duration_minutes,duration_seasons,audience_group,content_age_at_add
0,s1,TV Show,3%,Unknown,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,2020-08-14,2020,TV-MA,4,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...,2020,8,August,NaN,4.0,Adults,0
1,s10,Movie,1920,Vikram Bhatt,"Rajneesh Duggal, Adah Sharma, Indraneil Sengup...",India,2017-12-15,2008,TV-MA,143,"Horror Movies, International Movies, Thrillers",An architect and his wife move into a castle t...,2017,12,December,143.0,NaN,Adults,9
2,s100,Movie,3 Heroines,Iman Brotoseno,"Reza Rahadian, Bunga Citra Lestari, Tara Basro...",Indonesia,2019-01-05,2016,TV-PG,124,"Dramas, International Movies, Sports Movies",Three Indonesian women break records by becomi...,2019,1,January,124.0,NaN,Family,3
3,s1000,Movie,Blue Mountain State: The Rise of Thadland,Lev L. Spiro,"Alan Ritchson, Darin Brooks, James Cade, Rob R...",United States,2016-03-01,2016,R,90,Comedies,New NFL star Thad buys his old teammates' belo...,2016,3,March,90.0,NaN,Adults,0
4,s1001,TV Show,Blue Planet II,Unknown,David Attenborough,United Kingdom,2018-12-03,2017,TV-G,1,"British TV Shows, Docuseries, Science & Nature TV",This sequel to the award-winning nature series...,2018,12,December,NaN,1.0,Kids,1


In [17]:
## SAVE CLEANED MAIN CSV
df.to_csv("cleaned_netflix.csv", index=False)
print(f"\n  Saved cleaned_netflix.csv ({len(df):,} rows)")


  Saved cleaned_netflix.csv (7,682 rows)


In [18]:
df.columns

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'genres', 'description',
       'year_added', 'month_added', 'month_name', 'duration_minutes',
       'duration_seasons', 'audience_group', 'content_age_at_add'],
      dtype='object')

In [19]:
## GENRES EXPLODED
print("Exploding genres ...")
g = df[["show_id","type","title","release_year","year_added",
        "rating","audience_group","genres"]].copy()
g["genre"] = g["genres"].str.split(",")
g = g.explode("genre")
g["genre"] = g["genre"].str.strip()
g = g.drop(columns=["genres"])          
g.to_csv("genres_exploded.csv", index=False)
print(f"  Saved genres_exploded.csv ({len(g):,} rows)")

Exploding genres ...
  Saved genres_exploded.csv (16,859 rows)


In [20]:
g.head()

,show_id,type,title,release_year,year_added,rating,audience_group,genre
0,s1,TV Show,3%,2020,2020,TV-MA,Adults,International TV Shows
0,s1,TV Show,3%,2020,2020,TV-MA,Adults,TV Dramas
0,s1,TV Show,3%,2020,2020,TV-MA,Adults,TV Sci-Fi & Fantasy
1,s10,Movie,1920,2008,2017,TV-MA,Adults,Horror Movies
1,s10,Movie,1920,2008,2017,TV-MA,Adults,International Movies


In [21]:
## COUNTRIES EXPLODED
print("Exploding countries ...")
c = df[["show_id","type","title","release_year","year_added",
        "rating","country"]].copy()
c["country_single"] = c["country"].str.split(",")
c = c.explode("country_single")
c["country_single"] = c["country_single"].str.strip()
c = c[c["country_single"] != "Unknown"]
c.to_csv("countries_exploded.csv", index=False)
print(f"  Saved countries_exploded.csv ({len(c):,} rows)")

Exploding countries ...
  Saved countries_exploded.csv (8,954 rows)


In [22]:
## BUILD SUMMARY TABLES (for Excel sheets)print("\nBuilding summary tables ...")

yearly = (df.groupby(["year_added","type"])
            .size().reset_index())
yearly.rename(columns={0:'title_count'},inplace=True)


top_genres = g["genre"].value_counts().reset_index()
top_genres.columns = ["genre", "title_count"]

top_genres_yearly = (g.groupby(["genre","year_added"])
                      .size().reset_index(name="title_count"))

top_countries = c["country_single"].value_counts().reset_index()
top_countries.columns = ["country", "title_count"]

rating_dist = (df.groupby(["rating","audience_group","type"])
                 .size().reset_index(name="title_count")
                 .sort_values("title_count", ascending=False))

monthly = (df.groupby(["year_added","month_added","month_name"])
             .size().reset_index(name="title_count"))

top_directors = (df[df["director"] != "Unknown"]
                 .groupby(["director","type"])
                 .agg(title_count=("show_id","count"),
                      first_year=("release_year","min"),
                      latest_year=("release_year","max"))
                 .reset_index()
                 .sort_values("title_count", ascending=False)
                 .head(100))

kpi = pd.DataFrame([{
    "total_titles"   : len(df),
    "total_movies"   : int((df["type"]=="Movie").sum()),
    "total_tv_shows" : int((df["type"]=="TV Show").sum()),
    "movie_pct"      : round((df["type"]=="Movie").mean() * 100, 1),
    "tv_show_pct"    : round((df["type"]=="TV Show").mean() * 100, 1),
    "unique_ratings" : df["rating"].nunique(),
    "unique_genres"  : g["genre"].nunique(),
    "unique_countries": c["country_single"].nunique(),
    "year_min"       : int(df["year_added"].min()),
    "year_max"       : int(df["year_added"].max()),
    "avg_movie_mins" : round(df.loc[df["type"]=="Movie","duration_minutes"].mean(), 1),
}])



Building summary tables ...


In [23]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [24]:
## EXPORT TO EXCEL
print("Exporting to Excel ...")
with pd.ExcelWriter("netflix_for_excel.xlsx", engine="openpyxl") as writer:
    df.to_excel(writer,               sheet_name="Main_Data",          index=False)
    g.to_excel(writer,                sheet_name="Genres_Exploded",    index=False)
    c.to_excel(writer,                sheet_name="Countries_Exploded", index=False)
    yearly.to_excel(writer,           sheet_name="Yearly_Additions",   index=False)
    top_genres.to_excel(writer,       sheet_name="Top_Genres",         index=False)
    top_genres_yearly.to_excel(writer,sheet_name="Genres_Yearly",      index=False)
    top_countries.to_excel(writer,    sheet_name="Top_Countries",      index=False)
    rating_dist.to_excel(writer,      sheet_name="Rating_Dist",        index=False)
    monthly.to_excel(writer,          sheet_name="Monthly_Additions",  index=False)
    top_directors.to_excel(writer,    sheet_name="Top_Directors",      index=False)
    kpi.to_excel(writer,              sheet_name="KPI_Summary",        index=False)
print("  Saved netflix_for_excel.xlsx (11 sheets)")

Exporting to Excel ...
  Saved netflix_for_excel.xlsx (11 sheets)


In [25]:
## SUMMARY
print("\n=== STEP 1 COMPLETE ===")
print(f"Total clean titles  : {len(df):,}")
print(f"Movies              : {(df['type']=='Movie').sum():,}")
print(f"TV Shows            : {(df['type']=='TV Show').sum():,}")
print(f"Year range          : {int(df['year_added'].min())} - {int(df['year_added'].max())}")
print(f"Unique genres       : {g['genre'].nunique()}")
print(f"Unique countries    : {c['country_single'].nunique()}")
print("\nFiles created:")
print("  cleaned_netflix.csv")
print("  genres_exploded.csv")
print("  countries_exploded.csv")
print("  netflix_for_excel.xlsx  <- connect this to Power BI!")


=== STEP 1 COMPLETE ===
Total clean titles  : 7,682
Movies              : 5,372
TV Shows            : 2,310
Year range          : 2008 - 2021
Unique genres       : 42
Unique countries    : 118

Files created:
  cleaned_netflix.csv
  genres_exploded.csv
  countries_exploded.csv
  netflix_for_excel.xlsx  <- connect this to Power BI!
